In [1]:
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
from tqdm import tqdm
from torch.utils.data import DataLoader
from Define_Dataset_REG_Global import MRIDataset_REG
from SFCN_REG import SFCN_Regression

ROOT = Path(r"C:\Lab\test_retest_dataset")
OUTPUT_DIR = Path(r"C:\Lab\models\SFCN\reg\output")
MODEL_DIR = Path(r"C:\Lab\models\SFCN\reg") / "pretrained"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 1
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

final_min_coords = np.array([11, 12, 10])
final_max_coords = np.array([181, 221, 177])

MODEL_WEIGHTS = {
    "T1": MODEL_DIR / "pretrain_globalFOV_REG_T1_withcc_v2_20260226.pt",
    "GM": MODEL_DIR / "pretrain_globalFOV_REG_GMP_withcc_v2_20260314.pt",
    "WM": MODEL_DIR / "pretrain_globalFOV_REG_WMP_withcc_v1_20260306.pt",
}

PREFIX_MAP = {
    "T1": "rmi",
    "GM": "rp1",
    "WM": "rp2",
}

RUNS = [
    ("T1", "Maclaren"),
    ("T1", "MMRR"),
    ("T1", "OAS1"),

    ("WM", "Maclaren"),
    ("WM", "MMRR"),
    ("WM", "OAS1"),

    ("GM", "Maclaren"),
    ("GM", "MMRR"),
    ("GM", "OAS1"),
]

LABEL_FILES = {
    "Maclaren": ROOT / "Maclaren" / "Maclaren.xlsx",
    "MMRR": ROOT / "MMRR" / "MMRR.xlsx",
    "OAS1": ROOT / "OAS1" / "OAS1.xlsx",
}



In [2]:
model = SFCN_Regression(prediction_range=(5, 100)).to(device)

for modality, dataset_name in RUNS:

    print(f"\n===== {dataset_name} | {modality} =====")

    prefix = PREFIX_MAP[modality]
    img_route = ROOT / dataset_name / "ndf" / prefix

    # 讀取 Excel
    df = pd.read_excel(LABEL_FILES[dataset_name])

    test_df = df[["Subject_ID", "Age", prefix]].copy()
    test_df = test_df.rename(columns={prefix: "Filename"})
    test_df["Filename"] = test_df["Filename"].astype(str) + ".nii"

    # 建立 Dataset
    test_dataset = MRIDataset_REG(
        data_folders_set=[str(img_route)],
        labels_df=test_df[["Filename", "Age"]].reset_index(drop=True),
        crop_coords=(final_min_coords, final_max_coords)
    )

    test_loader = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=(device.type == "cuda")) # num_workers =0 4 差別?

    # 載入對應 Regression 模型
    model_path = MODEL_WEIGHTS[modality]

    model.load_state_dict(torch.load(model_path,map_location=device,weights_only=True))

    model.eval()

    all_true_ages = []
    all_pred_ages = []

    # 進行預測
    with torch.no_grad():

        for images, labels_hard in tqdm(test_loader,desc=f"{dataset_name}_{modality}"):

            images = images.float().to(device)    # 差別? images = images.detach().clone().float().to(device, non_blocking=True)
            labels_hard = labels_hard.float().to(device)   # 差別? labels_hard = labels_hard.detach().clone().float().to(device)
            outputs = model(images).view(-1)
            all_true_ages.extend(labels_hard.cpu().numpy().tolist())
            all_pred_ages.extend(outputs.cpu().numpy().tolist())


    # 檢查結果筆數
    if len(all_pred_ages) != len(test_df):
        raise ValueError(f"Prediction count mismatch: {dataset_name}/{modality}")

    # 整理預測結果
    result_df = pd.DataFrame({
        "Subject ID": test_df["Subject_ID"].to_numpy(),
        "True Age": all_true_ages,
        "Predicted Age": all_pred_ages
    })

    # 儲存 Excel
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_name = (f"SFCN_REG_{dataset_name}_{modality}_BN_{BATCH_SIZE}_{timestamp}.xlsx")
    output_path = OUTPUT_DIR / output_name
    result_df.to_excel(output_path, index=False)

    print(f"Samples: {len(result_df)}")
    print(f"Saved: {output_path}")

print("\nAll selected RUNS completed.")


===== Maclaren | T1 =====


Maclaren_T1: 100%|██████████| 120/120 [00:07<00:00, 15.62it/s]


Samples: 120
Saved: C:\Lab\models\SFCN\reg\output\SFCN_REG_Maclaren_T1_BN_1_20261014_221809.xlsx

===== MMRR | T1 =====


MMRR_T1: 100%|██████████| 42/42 [00:02<00:00, 16.13it/s]


Samples: 42
Saved: C:\Lab\models\SFCN\reg\output\SFCN_REG_MMRR_T1_BN_1_20261014_221812.xlsx

===== OAS1 | T1 =====


...

Samples: 120
Saved: C:\Lab\models\SFCN\reg\output\SFCN_REG_Maclaren_GM_BN_1_20261014_221834.xlsx

===== MMRR | GM =====


MMRR_GM: 100%|██████████| 42/42 [00:02<00:00, 17.22it/s]


Samples: 42
Saved: C:\Lab\models\SFCN\reg\output\SFCN_REG_MMRR_GM_BN_1_20261014_221836.xlsx

===== OAS1 | GM =====


OAS1_GM: 100%|██████████| 40/40 [00:02<00:00, 17.16it/s]

Samples: 40
Saved: C:\Lab\models\SFCN\reg\output\SFCN_REG_OAS1_GM_BN_1_20261014_221838.xlsx

All selected RUNS completed.


In [3]:

from pathlib import Path
import re
import pandas as pd
from datetime import datetime


# ============================================================
# 合併所有 SFCN Regression prediction Excel
# ============================================================

BATCH_SIZE = 1

OUTPUT_DIR = Path(r"C:\Lab\models\SFCN\reg\output")

output_file = OUTPUT_DIR / f"SFCN_REG_BN_{BATCH_SIZE}.xlsx"


# ------------------------------------------------------------
# 檔名格式：
# SFCN_REG_Maclaren_T1_BN_1_20261013_170000.xlsx
#
# 解析成：
# dataset = Maclaren
# model   = T1
# BN      = 1
# time    = 20261013_170000
# ------------------------------------------------------------

pattern = re.compile(
    r"^SFCN_REG_(Maclaren|MMRR|OAS1)_"
    rf"(T1|GM|WM)_BN_{BATCH_SIZE}_"
    r"(\d{8}_\d{6})\.xlsx$"
)


latest_files = {}

for file_path in OUTPUT_DIR.glob("SFCN_REG_*.xlsx"):

    # 排除合併結果
    if file_path.name == output_file.name:
        continue

    match = pattern.match(file_path.name)

    if match is None:
        continue

    dataset_name = match.group(1)
    model_type = match.group(2)
    timestamp_str = match.group(3)

    timestamp = datetime.strptime(
        timestamp_str,
        "%Y%m%d_%H%M%S"
    )

    sheet_name = f"{dataset_name}_{model_type}"

    # 同組合只保留最新結果
    if (
        sheet_name not in latest_files
        or timestamp > latest_files[sheet_name]["timestamp"]
    ):
        latest_files[sheet_name] = {
            "file": file_path,
            "timestamp": timestamp,
            "dataset": dataset_name,
            "model": model_type,
        }


# ============================================================
# 合併成單一 Excel，不同組合存成不同 Sheet
# ============================================================

if not latest_files:
    raise FileNotFoundError(
        f"No matching SFCN Regression prediction files in {OUTPUT_DIR}"
    )

with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    for sheet_name, info in latest_files.items():

        df = pd.read_excel(info["file"])

        # 只留下分析需要的欄位
        result_df = pd.DataFrame({
            "Dataset": info["dataset"],
            "Subject ID": df["Subject ID"],
            "True Age": df["True Age"],
            "Predicted Age": df["Predicted Age"],
        })

        result_df.to_excel(
            writer,
            sheet_name=sheet_name,
            index=False
        )

        print(
            f"{sheet_name:<25s} <- "
            f"{info['file'].name}"
        )


print(f"\n完成：{output_file}")

Maclaren_GM               <- SFCN_REG_Maclaren_GM_BN_1_20261014_221834.xlsx
Maclaren_T1               <- SFCN_REG_Maclaren_T1_BN_1_20261014_221809.xlsx
Maclaren_WM               <- SFCN_REG_Maclaren_WM_BN_1_20261014_221822.xlsx
MMRR_GM                   <- SFCN_REG_MMRR_GM_BN_1_20261014_221836.xlsx
MMRR_T1                   <- SFCN_REG_MMRR_T1_BN_1_20261014_221812.xlsx
MMRR_WM                   <- SFCN_REG_MMRR_WM_BN_1_20261014_221824.xlsx
OAS1_GM                   <- SFCN_REG_OAS1_GM_BN_1_20261014_221838.xlsx
OAS1_T1                   <- SFCN_REG_OAS1_T1_BN_1_20261014_221815.xlsx
OAS1_WM                   <- SFCN_REG_OAS1_WM_BN_1_20261014_221827.xlsx

完成：C:\Lab\models\SFCN\reg\output\SFCN_REG_BN_1.xlsx
